# Tool Calling

In [1]:
import dotenv
from agents import Agent, ModelSettings, Runner, function_tool, trace

dotenv.load_dotenv()

True

Create a static calorie table that we can use as a tool:

In [6]:
@function_tool
def get_food_calories(food_item: str) -> str:
    """
    Get calorie information for common foods to help with nutrition tracking.

    Args:
        food_item: Name of the food (e.g., "apple", "banana")

    Returns:
        Calorie information per standard serving
    """
    # Simple calorie database - in real world, you'd use USDA API
    calorie_data = {
        "apple": "80 calories per medium apple (182g)",
        "banana": "105 calories per medium banana (118g)",
        "broccoli": "25 calories per 1 cup chopped (91g)",
        "almonds": "164 calories per 1oz (28g) or about 23 nuts",
    }

    food_key = food_item.lower()
    if food_key in calorie_data:
        return f"{food_item.title()}: {calorie_data[food_key]}"
    else:
        return f"I don't have calorie data for {food_item} in my database. Try common foods like apple, chicken breast, or rice."

Let's test this out: 

_The following cell only works before you add the `@function_tool` annotation to `get_food_calories` function_

In [7]:
get_food_calories('banana')

TypeError: 'FunctionTool' object is not callable

In [4]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """)

In [5]:
with trace("Nutrition Assistant with tools"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )
    print(result.final_output)

About 200 calories total (roughly 105 kcal for a medium banana and 95 kcal for a medium apple).


Enforce tools use:

In [8]:
calorie_agent = Agent(
    name="Nutrition Assistant",
    instructions="""
    You are a helpful nutrition assistant giving out calorie information.
    You give concise answers.
    """,
    tools=[get_food_calories],
    model_settings=ModelSettings(tool_choice="get_food_calories"),
)

with trace("Nutrition Assistant with tools enforced"):
    result = await Runner.run(
        calorie_agent, "How many calories are in total in a banana and an apple?"
    )
    print(result.final_output)

Approximately 185 calories (105 for a medium banana + 80 for a medium apple). Note: actual calories vary with fruit size.


In [10]:
@function_tool
def get_food_nutrition(food_item: str) -> str:
    """Get approximate calories and macronutrients for a common food serving.

    Args:
        food_item: Name of the food (for example, "apple" or "banana").

    Returns:
        Approximate calories, protein, carbohydrates, fat, and fiber.
    """
    nutrition_data = {
        "apple": {
            "serving": "1 medium apple (182g)",
            "calories": 95,
            "protein": 0.5,
            "carbs": 25.1,
            "fat": 0.3,
            "fiber": 4.4,
        },
        "banana": {
            "serving": "1 medium banana (118g)",
            "calories": 105,
            "protein": 1.3,
            "carbs": 27.0,
            "fat": 0.4,
            "fiber": 3.1,
        },
        "broccoli": {
            "serving": "1 cup chopped raw broccoli (91g)",
            "calories": 31,
            "protein": 2.5,
            "carbs": 6.0,
            "fat": 0.3,
            "fiber": 2.4,
        },
        "almonds": {
            "serving": "1 ounce (28g)",
            "calories": 164,
            "protein": 6.0,
            "carbs": 6.1,
            "fat": 14.2,
            "fiber": 3.5,
        },
    }

    food_key = food_item.strip().lower()
    if food_key not in nutrition_data:
        return (
            f"I don't have nutrition data for {food_item}. "
            "Try apple, banana, broccoli, or almonds."
        )

    details = nutrition_data[food_key]
    return (
        f"Approximate nutrition for {food_key.title()} "
        f"({details['serving']}): {details['calories']} calories, "
        f"{details['protein']}g protein, {details['carbs']}g carbohydrates, "
        f"{details['fat']}g fat, and {details['fiber']}g fiber."
    )


nutrition_agent = Agent(
    name="Nutrition Details Assistant",
    instructions="""
    You provide concise nutrition details for foods using the get_food_nutrition tool.
    Give the serving size and report the values as approximate estimates.
    If a food is not in the tool's dataset, say so instead of guessing.
    Do not give medical advice.
    """,
    tools=[get_food_nutrition],
    model_settings=ModelSettings(tool_choice="get_food_nutrition"),
)

with trace("Nutrition Details Assistant with tools"):
    result = await Runner.run(
        nutrition_agent, "What nutrition is in a banana?"
    )
    print(result.final_output)

Banana (1 medium, about 118 g) — approximate per serving:
- Calories: 105
- Protein: 1.3 g
- Carbohydrates: 27 g
- Fat: 0.4 g
- Fiber: 3.1 g
